# SteerMoE: forcing number formatting on Qwen3-30B-A3B

Replicates **"Steering MoE LLMs via Expert (De)Activation"** ([arXiv:2509.09660](https://arxiv.org/abs/2509.09660), [official code](https://github.com/adobe-research/SteerMoE)) on **Qwen3-30B-A3B** (48 MoE layers × 128 experts, top-8), one of the models evaluated in the paper, end to end in one engine:

1. **Detection** — per-token router logits are captured for contrastive pairs (answering with digits `1, 2, 3` vs. words `one, two, three`) with EasySteer's `router_logits` capture stream, each expert's top-k selection rate on the behavior tokens yields the risk difference `Δ = p_digits − p_words`, and the top word-linked experts are saved as a `deactivate` steering config (`steermoe_qwen3_words.json`).
2. **Steering** — compare greedy counting with and without deactivating those experts, then count any deactivated-expert selections in the steered router outputs.

Qwen3-30B-A3B is a hybrid thinking model; prompts render with `enable_thinking=False`, as in the official SteerMoE code.

**Execution note:** Outputs are cleared after the API migration. Run the notebook from top to bottom to obtain results for your environment.


In [ ]:
import json
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("VLLM_LOGGING_LEVEL", "WARNING")  # quiet engine boot logs

import numpy as np
from easysteer.capture import capture
from vllm import LLM, SamplingParams
from vllm.capture import SelectSpec
from vllm.steer_vectors import ApplySpec, SteeringSpec, VectorSpec
from transformers import AutoConfig

MODEL = os.environ.get("EASYSTEER_MODEL", "Qwen/Qwen3-30B-A3B")
hf_cfg = AutoConfig.from_pretrained(MODEL).to_dict()
N_EXPERTS = hf_cfg["num_experts"]      # 128
TOP_K = hf_cfg["num_experts_per_tok"]  # 8

llm = LLM(
    model=MODEL,
    tensor_parallel_size=int(os.environ.get("EASYSTEER_TP", "1")),
    enable_steer_vector=True,
    steer_algorithms=["moe_router"],
    # This notebook only uses deactivate, which supports in-graph steering.
    steer_graph_mode="in_graph",
    gpu_memory_utilization=0.92,
    max_model_len=4096,
    max_num_seqs=4,
)
tok = llm.get_tokenizer()

## Expert detection

### The contrastive pairs

Each side renders a full chat turn **including the assistant response**, so
a single prefill routes every response token through the MoE layers. The
`target` string marks the tokens whose routings we compare. The official
demo uses a single pair; a few pairs sharpen the risk difference
considerably.

In [ ]:
PAIRS = [
    ("Count to ten",
     "1, 2, 3, 4, 5, 6, 7, 8, 9, 10",
     "one, two, three, four, five, six, seven, eight, nine, ten"),
    ("How many days are in a week, and how many months in a year?",
     "There are 7 days in a week and 12 months in a year.",
     "There are seven days in a week and twelve months in a year."),
    ("What is five plus three?",
     "5 + 3 = 8",
     "five plus three equals eight"),
]

### Capture router logits

`capture(..., stream="router_logits")` returns labeled rows grouped by
sample and true model layer ID. Select the answer's token positions at
capture time; the helper manages the capture lifecycle while the engine
handles graph execution and prefix-cache reads.

In [ ]:
def find_sub_list(sub, seq):
    n = len(sub)
    return [(i, i + n - 1) for i in range(len(seq) - n + 1)
            if seq[i:i + n] == sub]


def topk_membership(rows):
    """(tokens, n_experts) logits -> bool top-k membership mask."""
    order = np.argsort(rows, axis=-1)[:, -TOP_K:]
    mask = np.zeros(rows.shape, dtype=bool)
    np.put_along_axis(mask, order, True, axis=-1)
    return mask


counts = {"digits": None, "words": None}
totals = {"digits": 0, "words": 0}
layer_ids = None
for user, digits_ans, words_ans in PAIRS:
    for key, answer in (("digits", digits_ans), ("words", words_ans)):
        messages = [{"role": "user", "content": user},
                {"role": "assistant", "content": answer}]
        prompt_ids = tok.apply_chat_template(
            messages, tokenize=True, return_dict=False, add_generation_prompt=False,
            enable_thinking=False,
        )
        prompt = {"prompt_token_ids": prompt_ids}
        target_ids = tok(answer, add_special_tokens=False).input_ids
        s, e = find_sub_list(target_ids, prompt_ids)[-1]
        result = capture(
            llm, [prompt],
            stream="router_logits",
            select=SelectSpec(prompt_window=(s, e + 1)),
            steering=False,
        )
        if layer_ids is None:
            layer_ids = result.layer_ids
        if result.layer_ids != layer_ids:
            raise RuntimeError("captured MoE layer IDs changed between samples")
        if result.sample_positions(0) != list(range(s, e + 1)):
            raise RuntimeError("capture did not return every target token position")
        logits = result.sample(0)

        # Statistics use array rows; layer_ids maps them to model layer IDs.
        sel = np.stack([topk_membership(logits[lid].float().numpy())
                        for lid in layer_ids])
        cnt = sel.sum(axis=1)  # (layer row, expert)
        counts[key] = cnt if counts[key] is None else counts[key] + cnt
        totals[key] += e - s + 1

print(f"detection tokens: digits={totals['digits']} "
      f"words={totals['words']}")

### Risk difference

`Δ(layer, expert) = p_digits − p_words`: experts with large positive Δ are
selected for digit tokens but not word tokens.

In [ ]:
risk_diff = counts["digits"] / totals["digits"] \
    - counts["words"] / totals["words"]

flat = np.argsort(np.abs(risk_diff), axis=None)[::-1]
print("top behavior-linked experts (layer, expert, Δ):")
for idx in flat[:10]:
    row, expert = divmod(int(idx), N_EXPERTS)
    layer = layer_ids[row]
    print(f"  L{layer:02d} E{expert:02d}  "
          f"Δ={risk_diff[row, expert]:+.2f}")

### Save the steering config

Deactivating the **word-linked** experts (negative Δ) steers away from
spelled-out numbers. Keep the experiment's 200 deactivated experts
(~3% of 128×48); rerun the comparisons below to measure the shift
toward digits. The paper tunes this count per model and task (Table A.2).
The config uses true model layer IDs, even when captured layers are
non-contiguous.

In [ ]:
N_DEACT = 200

deact = {}
taken = 0
for idx in flat:
    row, expert = divmod(int(idx), N_EXPERTS)
    layer = layer_ids[row]
    # word-linked experts have negative delta (digits_rate - words_rate)
    if risk_diff[row, expert] >= 0:
        continue
    deact.setdefault(layer, []).append(expert)
    taken += 1
    if taken == N_DEACT:
        break

with open("steermoe_qwen3_words.json", "w") as f:
    json.dump({"layer_configs": {
        str(layer): {"mode": "deactivate", "expert_ids": ids}
        for layer, ids in deact.items()
    }}, f, indent=2)
print(f"saved steermoe_qwen3_words.json: {taken} experts "
      f"across {len(deact)} layers")

## Steering

In [ ]:
with open("steermoe_qwen3_words.json") as f:
    layer_configs = json.load(f)["layer_configs"]

steering = SteeringSpec(vectors=[
    VectorSpec(
        source="steermoe_qwen3_words.json",
        algorithm="moe_router",  # per-layer mode/expert_ids come from the JSON
        layers=sorted(int(la) for la in layer_configs),
        apply=ApplySpec(prompt="all", generation="all"),
    ),
])


def make_prompt(user_text):
    messages = [{"role": "user", "content": user_text}]
    prompt_ids = tok.apply_chat_template(
        messages, tokenize=True, return_dict=False,
        add_generation_prompt=True, enable_thinking=False,
    )
    return {"prompt_token_ids": prompt_ids}


def gen(user_text, spec=False):
    prompt = make_prompt(user_text)
    out = llm.generate(prompt,
                       sampling_params=SamplingParams(temperature=0.0,
                                                      max_tokens=64),
                       steering=spec,
                       use_tqdm=False)
    return out[0].outputs[0].text.strip().replace("\n", " ")


def digit_share(text):
    digits = sum(c.isdigit() for c in text)
    letters = sum(c.isalpha() for c in text)
    return digits / max(1, digits + letters)

In [ ]:
PROMPTS = [
    "Count to fifteen.",
    "Count from one to twelve.",
]

for prompt in PROMPTS:
    base = gen(prompt)
    steered = gen(prompt, steering)
    print(f"[{prompt}]")
    print(f"  baseline (digit-share {digit_share(base):.2f}): {base}")
    print(f"  steered  (digit-share {digit_share(steered):.2f}): {steered}")

### Mechanism check

Capture the post-steering router logits for one steered prompt and
count deactivated experts in each token's top-8. A count of zero confirms
that no deactivated expert was selected in these captured rows.

In [ ]:
deact = {int(la): c["expert_ids"] for la, c in layer_configs.items()}

prompt = make_prompt("Count to fifteen.")
result = capture(
    llm, [prompt],
    max_tokens=32,
    stream="router_logits",
    select=SelectSpec(prompt="all", generation="all"),
    steering=steering,
)
logits = {lid: tensor.float().numpy()
          for lid, tensor in result.sample(0).items()}

leaks = 0
for layer, expert_ids in deact.items():
    top = np.argsort(logits[layer], axis=-1)[:, -TOP_K:]
    leaks += int(np.isin(top, expert_ids).sum())
print(f"deactivated-expert selections post-steering: {leaks} "
      f"(0 = steering is airtight)")